# Parte 3: baseline BM25 x denso x HyDE (SciFact)

Responsável: Kossi Sedjro Mawuli Dominique Houessou

Seminários de Computação I. Artigo base: HyDE (Gao et al., 2022), arXiv 2212.10496.

Este notebook reproduz, em escala reduzida, a comparação central do artigo na base SciFact (BEIR). Três métodos de recuperação são comparados:

1. **BM25**: busca lexical clássica.
2. **Busca densa**: a pergunta é codificada em um vetor e comparada com os vetores dos documentos.
3. **HyDE**: um LLM escreve passagens hipotéticas que respondem à pergunta, e a busca usa a média dos vetores da pergunta e dessas passagens (mesma regra do `hyde.py` do repositório texttron/hyde).

Os métodos denso e HyDE são rodados com dois encoders: o **Contriever** (o do artigo, em inglês) e o **multilingual-e5-small** (que também serve para o português na parte do Quati). Métricas: nDCG@10 e Recall@100.

**Como rodar:** Ambiente de execução > Alterar tipo de ambiente de execução > GPU T4. Rode as células em ordem. Na primeira vez, use `N_QUERIES = 30` na célula de configuração para testar rápido, e depois volte para `None` (as 300 consultas) para o resultado final.

In [ ]:
!pip -q install rank_bm25 sentence-transformers
!nvidia-smi -L || echo "ATENÇÃO: sem GPU. Ative a GPU em Ambiente de execução > Alterar tipo de ambiente de execução."

## 1. Configuração

In [ ]:
import os, re, json, time, zipfile, urllib.request
import numpy as np
import pandas as pd

DATASET = "scifact"
N_QUERIES = None        # use 30 na primeira execução; None usa todas as consultas de teste (300)
TOP_K = 100

# normalize=True usa cosseno; normalize=False usa produto escalar (como o Contriever original)
ENCODERS = [
    {"nome": "Contriever", "modelo": "facebook/contriever",
     "q_prefix": "", "d_prefix": "", "normalize": False},
    {"nome": "mE5-small", "modelo": "intfloat/multilingual-e5-small",
     "q_prefix": "query: ", "d_prefix": "passage: ", "normalize": True},
]

# O artigo usa text-davinci-003 com 8 passagens de até 512 tokens. Aqui usamos um LLM aberto pequeno.
LLM_NOME = "Qwen/Qwen2.5-1.5B-Instruct"
N_HIPOTESES = 4
MAX_NEW_TOKENS = 200
TEMPERATURA = 0.7
SEED = 42

# Prompt da tarefa SciFact, igual ao do arquivo promptor.py do repositório texttron/hyde
PROMPT = "Please write a scientific paper passage to support/refute the claim.\nClaim: {}\nPassage:"

## 2. Dados (BEIR)

In [ ]:
BEIR_URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip"


def download_beir(nome, destino="beir_data"):
    caminho = os.path.join(destino, nome)
    if os.path.exists(os.path.join(caminho, "corpus.jsonl")):
        return caminho
    os.makedirs(destino, exist_ok=True)
    zip_path = os.path.join(destino, f"{nome}.zip")
    if not os.path.exists(zip_path):
        try:
            urllib.request.urlretrieve(BEIR_URL.format(nome), zip_path)
        except Exception as e:
            raise RuntimeError(
                f"Falha ao baixar {BEIR_URL.format(nome)} ({e}). Baixe o zip pelo navegador, "
                f"envie para a pasta '{destino}' do Colab (painel de arquivos) e rode a célula de novo.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(destino)
    return caminho


def read_beir(caminho, split="test"):
    corpus = {}
    with open(os.path.join(caminho, "corpus.jsonl"), encoding="utf-8") as f:
        for linha in f:
            d = json.loads(linha)
            corpus[d["_id"]] = {"title": d.get("title", ""), "text": d["text"]}
    queries = {}
    with open(os.path.join(caminho, "queries.jsonl"), encoding="utf-8") as f:
        for linha in f:
            d = json.loads(linha)
            queries[d["_id"]] = d["text"]
    qrels = {}
    with open(os.path.join(caminho, "qrels", f"{split}.tsv"), encoding="utf-8") as f:
        next(f)  # cabeçalho
        for linha in f:
            qid, did, score = linha.rstrip("\n").split("\t")
            if int(score) > 0:
                qrels.setdefault(qid, {})[did] = int(score)
    queries = {q: t for q, t in queries.items() if q in qrels}
    return corpus, queries, qrels


def texto_doc(d):
    return (d["title"] + " " + d["text"]).strip()

In [ ]:
corpus, queries, qrels = read_beir(download_beir(DATASET))
doc_ids = list(corpus)
doc_textos = [texto_doc(corpus[d]) for d in doc_ids]

qids = list(queries)
if N_QUERIES:
    qids = qids[:N_QUERIES]

print(f"Documentos: {len(doc_ids)} | Consultas avaliadas: {len(qids)}")
print("Exemplo de consulta:", queries[qids[0]])

## 3. Métricas

In [ ]:
def ndcg_at_k(ranking, rels, k=10):
    dcg = sum(rels.get(d, 0) / np.log2(i + 2) for i, d in enumerate(ranking[:k]))
    ideal = sorted(rels.values(), reverse=True)[:k]
    idcg = sum(r / np.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def recall_at_k(ranking, rels, k=100):
    relevantes = set(rels)
    return len(relevantes & set(ranking[:k])) / len(relevantes)


def avaliar(rankings, qrels):
    qs = list(rankings)
    return {
        "nDCG@10": float(np.mean([ndcg_at_k(rankings[q], qrels[q], 10) for q in qs])),
        "Recall@100": float(np.mean([recall_at_k(rankings[q], qrels[q], 100) for q in qs])),
    }

## 4. Funções que usam modelos

Ficam isoladas aqui: encoder (Sentence-Transformers) e gerador de passagens hipotéticas (LLM aberto).

In [ ]:
def load_encoder(spec):
    import torch
    from sentence_transformers import SentenceTransformer
    device = "cuda" if torch.cuda.is_available() else "cpu"
    return SentenceTransformer(spec["modelo"], device=device)


def encode_texts(modelo, textos, prefixo="", normalize=False, batch_size=64):
    entrada = [prefixo + t for t in textos]
    return modelo.encode(entrada, batch_size=batch_size, convert_to_numpy=True,
                         normalize_embeddings=normalize, show_progress_bar=False)


def load_llm(nome):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(nome)
    tok.padding_side = "left"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32
    modelo = AutoModelForCausalLM.from_pretrained(nome, torch_dtype=dtype).to(device)
    modelo.eval()
    return modelo, tok


def generate_hypotheses(llm, prompts, n, max_new_tokens, temperatura, batch_size=16, seed=42):
    import torch
    from tqdm.auto import tqdm
    modelo, tok = llm
    torch.manual_seed(seed)
    resultado = []
    for i in tqdm(range(0, len(prompts), batch_size), desc="Gerando passagens"):
        lote = prompts[i:i + batch_size]
        textos = [tok.apply_chat_template([{"role": "user", "content": p}],
                                          tokenize=False, add_generation_prompt=True) for p in lote]
        entrada = tok(textos, return_tensors="pt", padding=True).to(modelo.device)
        with torch.no_grad():
            saida = modelo.generate(
                **entrada, do_sample=True, temperature=temperatura, top_p=1.0, top_k=0,
                max_new_tokens=max_new_tokens, num_return_sequences=n,
                pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)
        novos = saida[:, entrada["input_ids"].shape[1]:]
        decodificados = tok.batch_decode(novos, skip_special_tokens=True)
        for j in range(len(lote)):
            resultado.append([d.strip() for d in decodificados[j * n:(j + 1) * n]])
    return resultado

## 5. Funções de busca

In [ ]:
from rank_bm25 import BM25Okapi


def tokenizar(s):
    return re.findall(r"\w+", s.lower())


def bm25_rank(bm25, qids, queries, doc_ids, k=TOP_K):
    ranking = {}
    for q in qids:
        scores = bm25.get_scores(tokenizar(queries[q]))
        topo = np.argsort(-scores)[:k]
        ranking[q] = [doc_ids[i] for i in topo]
    return ranking


def rank_from_vectors(V, doc_embs, doc_ids, qids, k=TOP_K):
    scores = V @ doc_embs.T
    topo = np.argsort(-scores, axis=1)[:, :k]
    return {q: [doc_ids[i] for i in topo[n]] for n, q in enumerate(qids)}


def hyde_vectors(modelo, spec, qids, queries, hip):
    # Regra do hyde.py: média dos vetores de [pergunta] + passagens hipotéticas.
    # A pergunta usa o prefixo de consulta; as passagens hipotéticas usam o prefixo de documento.
    n = len(hip[qids[0]])
    assert all(len(hip[q]) == n for q in qids), "Número de passagens diferente entre consultas"
    Q = encode_texts(modelo, [queries[q] for q in qids], spec["q_prefix"], spec["normalize"])
    plano = [h for q in qids for h in hip[q]]
    H = encode_texts(modelo, plano, spec["d_prefix"], spec["normalize"])
    H = H.reshape(len(qids), n, -1)
    return (Q + H.sum(axis=1)) / (n + 1)

## 6. BM25 e busca densa (rápidos; rodam antes da geração para falhar cedo)

In [ ]:
resultados = []
rankings = {}


def registrar(metodo, ranking, t_busca, t_geracao=None):
    m = avaliar(ranking, qrels)
    resultados.append({"Método": metodo,
                       "nDCG@10": round(m["nDCG@10"], 4),
                       "Recall@100": round(m["Recall@100"], 4),
                       "Tempo de busca (s)": round(t_busca, 1),
                       "Geração LLM (s)": None if t_geracao is None else round(t_geracao, 1)})
    rankings[metodo] = ranking
    print(f"{metodo:<22} nDCG@10={m['nDCG@10']:.4f}  Recall@100={m['Recall@100']:.4f}")


t0 = time.time()
bm25 = BM25Okapi([tokenizar(t) for t in doc_textos])
rk = bm25_rank(bm25, qids, queries, doc_ids)
registrar("BM25", rk, time.time() - t0)

encoders = {}
doc_embs = {}
for spec in ENCODERS:
    modelo = load_encoder(spec)
    encoders[spec["nome"]] = modelo
    t0 = time.time()
    doc_embs[spec["nome"]] = encode_texts(modelo, doc_textos, spec["d_prefix"], spec["normalize"])
    Q = encode_texts(modelo, [queries[q] for q in qids], spec["q_prefix"], spec["normalize"])
    rk = rank_from_vectors(Q, doc_embs[spec["nome"]], doc_ids, qids)
    registrar(f"Denso ({spec['nome']})", rk, time.time() - t0)

## 7. Geração das passagens hipotéticas (LLM)

O resultado é salvo em um arquivo JSON. Se o arquivo existir, a geração não é repetida (útil se a sessão do Colab cair).

In [ ]:
cache = f"hipoteses_{LLM_NOME.replace('/', '_')}_{N_HIPOTESES}_{len(qids)}.json"

if os.path.exists(cache):
    with open(cache, encoding="utf-8") as f:
        hip = json.load(f)
    t_geracao = None
    print("Passagens carregadas do cache:", cache)
else:
    llm = load_llm(LLM_NOME)
    t0 = time.time()
    saida = generate_hypotheses(llm, [PROMPT.format(queries[q]) for q in qids],
                                N_HIPOTESES, MAX_NEW_TOKENS, TEMPERATURA, seed=SEED)
    t_geracao = time.time() - t0
    # se o LLM devolver texto vazio, usa a própria consulta como passagem
    hip = {q: [h if h else queries[q] for h in hs] for q, hs in zip(qids, saida)}
    with open(cache, "w", encoding="utf-8") as f:
        json.dump(hip, f, ensure_ascii=False)
    print(f"Geração concluída em {t_geracao:.0f} s")

## 8. HyDE

In [ ]:
for spec in ENCODERS:
    modelo = encoders[spec["nome"]]
    t0 = time.time()
    V = hyde_vectors(modelo, spec, qids, queries, hip)
    rk = rank_from_vectors(V, doc_embs[spec["nome"]], doc_ids, qids)
    registrar(f"HyDE ({spec['nome']})", rk, time.time() - t0, t_geracao)

## 9. Tabela de resultados

In [ ]:
tabela = pd.DataFrame(resultados)
tabela.to_csv("resultados_baseline_scifact.csv", index=False)
tabela

## 9b. A diferença é estatisticamente confiável?

A média do nDCG@10 esconde a variação entre consultas. Aqui calculamos o nDCG@10 de cada consulta e comparamos HyDE contra a busca densa com a mesma consulta nos dois lados (teste pareado). Mostramos a diferença média, um intervalo de confiança de 95% por bootstrap (reamostragem das consultas) e os p-valores do teste t pareado e do teste de Wilcoxon. Se o intervalo não contém o zero, a vantagem não é só ruído da amostra de consultas.

In [ ]:
from scipy import stats

def ndcg_por_consulta(ranking):
    return np.array([ndcg_at_k(ranking[q], qrels[q], 10) for q in qids])

por_consulta = {nome: ndcg_por_consulta(rk) for nome, rk in rankings.items()}


def comparar(a, b, n_boot=10000, seed=SEED):
    """Compara o método a contra o método b (diferença a - b), consulta a consulta."""
    d = por_consulta[a] - por_consulta[b]
    rng = np.random.default_rng(seed)
    medias = rng.choice(d, size=(n_boot, len(d)), replace=True).mean(axis=1)
    lo, hi = np.percentile(medias, [2.5, 97.5])
    p_t = stats.ttest_1samp(d, 0.0).pvalue if d.std() > 0 else 1.0
    try:
        p_w = stats.wilcoxon(d).pvalue
    except ValueError:      # todas as diferenças são zero
        p_w = 1.0
    return {"Comparação": f"{a} - {b}", "Diferença média": d.mean(),
            "IC 95% inferior": lo, "IC 95% superior": hi,
            "p (t pareado)": p_t, "p (Wilcoxon)": p_w,
            "Consultas HyDE melhor": int((d > 0).sum()),
            "Consultas pior": int((d < 0).sum()), "Empates": int((d == 0).sum())}


pares = []
for spec in ENCODERS:
    n = spec["nome"]
    pares.append((f"HyDE ({n})", f"Denso ({n})"))
    pares.append((f"HyDE ({n})", "BM25"))

tabela_sig = pd.DataFrame([comparar(a, b) for a, b in pares])
tabela_sig.to_csv("significancia_baseline_scifact.csv", index=False)
tabela_sig.round(4)

## 10. Exemplo qualitativo (bom para os slides)

Escolhemos automaticamente a consulta em que o HyDE mais melhora o nDCG@10 em relação à busca densa (primeiro encoder). Assim o exemplo mostra um caso em que a passagem hipotética ajuda a encontrar o documento certo. Logo abaixo está também um caso em que o HyDE piora, para a discussão de limitações.

In [ ]:
def titulo(d):
    return corpus[d]["title"][:80]


def mostrar(qid, enc_nome):
    print("Afirmação:", queries[qid])
    print("\nPassagem hipotética (1ª):\n", hip[qid][0][:600])
    print("\nDocumentos relevantes:", [titulo(d) for d in qrels[qid]])
    for nome in (f"Denso ({enc_nome})", f"HyDE ({enc_nome})"):
        ranking = rankings[nome]
        top3 = [("[ok] " if d in qrels[qid] else "[x] ") + titulo(d) for d in ranking[qid][:3]]
        pos = [i + 1 for i, d in enumerate(ranking[qid]) if d in qrels[qid]]
        print(f"\n{nome} (nDCG@10 = {ndcg_at_k(ranking[qid], qrels[qid], 10):.2f}, "
              f"posição do 1º relevante: {pos[0] if pos else 'fora do top ' + str(TOP_K)}):", *top3, sep="\n  ")


enc_ex = ENCODERS[0]["nome"]
ganho = por_consulta[f"HyDE ({enc_ex})"] - por_consulta[f"Denso ({enc_ex})"]
i_ganha, i_perde = int(np.argmax(ganho)), int(np.argmin(ganho))

print("=" * 70, "\nCASO EM QUE O HyDE GANHA (ganho de nDCG@10 = %+.2f)\n" % ganho[i_ganha], "=" * 70, sep="")
mostrar(qids[i_ganha], enc_ex)
rotulo = "CASO EM QUE O HyDE PERDE" if ganho[i_perde] < 0 else "CASO EM QUE O HyDE MENOS AJUDA (nenhuma consulta piorou)"
print("\n" + "=" * 70, "\n%s (variação de nDCG@10 = %+.2f)\n" % (rotulo, ganho[i_perde]), "=" * 70, sep="")
mostrar(qids[i_perde], enc_ex)

## 11. Ablações: quantas passagens e o papel da consulta

Duas perguntas sobre o HyDE, respondidas com os mesmos dados da baseline:

1. **Quantas passagens hipotéticas são necessárias?** Usamos as primeiras 1, 2, 3 e 4 passagens já geradas (nenhuma geração nova). A linha com 0 passagens é a busca densa simples.
2. **A própria consulta deve entrar na média do vetor?** O artigo inclui a consulta. Aqui comparamos "com a consulta" e "só as passagens".

Cada linha mostra o nDCG@10, o Recall@100 e a diferença para a busca densa, com intervalo de confiança de 95% por bootstrap. Se o intervalo contém o zero, a diferença não é confiável.

A célula 11a abaixo é **opcional**: gera mais 4 passagens (leva cerca de 5 minutos na T4) para testar até 8 passagens, como no artigo. Para pular, deixe `GERAR_MAIS = False`.

In [ ]:
GERAR_MAIS = False      # True gera mais 4 passagens (total 8, como no artigo); leva cerca de 5 min

if GERAR_MAIS:
    cache2 = f"hipoteses_extra_{LLM_NOME.replace('/', '_')}_{N_HIPOTESES}_{len(qids)}.json"
    if os.path.exists(cache2):
        with open(cache2, encoding="utf-8") as f:
            extra = json.load(f)
    else:
        llm = load_llm(LLM_NOME)
        # semente diferente da primeira geração, para as novas passagens não repetirem as antigas
        saida = generate_hypotheses(llm, [PROMPT.format(queries[q]) for q in qids],
                                    N_HIPOTESES, MAX_NEW_TOKENS, TEMPERATURA, seed=SEED + 1)
        extra = {q: [h if h else queries[q] for h in hs] for q, hs in zip(qids, saida)}
        with open(cache2, "w", encoding="utf-8") as f:
            json.dump(extra, f, ensure_ascii=False)
    hip8 = {q: hip[q] + extra[q] for q in qids}
    print("Passagens por consulta:", len(hip8[qids[0]]))
else:
    print("Ablação usará as", len(hip[qids[0]]), "passagens já geradas.")

In [ ]:
import matplotlib.pyplot as plt

hip_abl = globals().get("hip8", hip)
n_max = len(hip_abl[qids[0]])
rng = np.random.default_rng(SEED)
idx = rng.integers(0, len(qids), size=(5000, len(qids)))      # mesmas reamostragens para todas as linhas

linhas = []
for spec in ENCODERS:
    nome, modelo = spec["nome"], encoders[spec["nome"]]
    Q = encode_texts(modelo, [queries[q] for q in qids], spec["q_prefix"], spec["normalize"])
    plano = [h for q in qids for h in hip_abl[q]]
    H = encode_texts(modelo, plano, spec["d_prefix"], spec["normalize"]).reshape(len(qids), n_max, -1)

    rk_base = rankings[f"Denso ({nome})"]
    base = np.array([ndcg_at_k(rk_base[q], qrels[q], 10) for q in qids])
    m0 = avaliar(rk_base, qrels)
    linhas.append({"Encoder": nome, "Passagens": 0, "Consulta no vetor": "sim",
                   "nDCG@10": m0["nDCG@10"], "Recall@100": m0["Recall@100"],
                   "Dif. vs denso": 0.0, "IC 95% inf.": 0.0, "IC 95% sup.": 0.0})

    for inc in (1, 0):
        for n in range(1, n_max + 1):
            V = (Q * inc + H[:, :n].sum(axis=1)) / (n + inc)
            rk = rank_from_vectors(V, doc_embs[nome], doc_ids, qids)
            m = avaliar(rk, qrels)
            nd = np.array([ndcg_at_k(rk[q], qrels[q], 10) for q in qids])
            d = nd - base
            lo, hi = np.percentile(d[idx].mean(axis=1), [2.5, 97.5])
            linhas.append({"Encoder": nome, "Passagens": n,
                           "Consulta no vetor": "sim" if inc else "não",
                           "nDCG@10": m["nDCG@10"], "Recall@100": m["Recall@100"],
                           "Dif. vs denso": d.mean(), "IC 95% inf.": lo, "IC 95% sup.": hi})
            # conferência: a configuração do artigo deve reproduzir a tabela principal
            if inc == 1 and n == len(hip[qids[0]]) and hip_abl is hip:
                ref = avaliar(rankings[f"HyDE ({nome})"], qrels)["nDCG@10"]
                assert abs(m["nDCG@10"] - ref) < 1e-6, "ablação não reproduz o HyDE principal"

tabela_abl = pd.DataFrame(linhas)
tabela_abl.to_csv("ablacoes_baseline_scifact.csv", index=False)

fig, eixos = plt.subplots(1, len(ENCODERS), figsize=(5.5 * len(ENCODERS), 4), sharey=False)
eixos = np.atleast_1d(eixos)
for ax, spec in zip(eixos, ENCODERS):
    t = tabela_abl[tabela_abl["Encoder"] == spec["nome"]]
    base_val = t[t["Passagens"] == 0]["nDCG@10"].iloc[0]
    for rot in ("sim", "não"):
        s = t[(t["Passagens"] > 0) & (t["Consulta no vetor"] == rot)]
        ax.plot(s["Passagens"], s["nDCG@10"], marker="o", label=f"HyDE, consulta no vetor: {rot}")
    ax.axhline(base_val, color="gray", linestyle="--", label="Denso (sem HyDE)")
    ax.set_title(spec["nome"]); ax.set_xlabel("Passagens hipotéticas"); ax.set_ylabel("nDCG@10")
    ax.set_xticks(range(1, n_max + 1)); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

tabela_abl.round(4)

## Notas para o relatório

- **O que comparar com o artigo:** a tendência (HyDE acima da busca densa sem HyDE), não o valor absoluto. O artigo usa um LLM muito maior (text-davinci-003) e 8 passagens por consulta; aqui há um LLM de 1,5 bilhão de parâmetros e 4 passagens. Confira a tabela de resultados do artigo antes de escrever a discussão.
- **Adaptação para o mE5:** o mE5 usa prefixos ("query: " e "passage: "). A pergunta é codificada como consulta e as passagens hipotéticas como documentos. Essa escolha não existe no artigo e deve constar como decisão metodológica.
- **BM25:** esta versão (`rank_bm25`, sem stemming nem stopwords) pode diferir um pouco do BM25 usado nos artigos (Elasticsearch ou Pyserini).
- **Tempo:** o tempo de busca do HyDE não inclui a geração das passagens, que aparece na coluna "Geração LLM (s)". Se o resultado veio do cache, essa coluna fica vazia.
- **Significância:** a seção 9b mostra a diferença média por consulta, o intervalo de confiança por bootstrap e os p-valores. Relate o intervalo, não só o p-valor, e diga que são 300 consultas de uma única base.
- **Ablações:** a seção 11 usa só as passagens já geradas (até 4). Para 8 passagens, como no artigo, mude `GERAR_MAIS` para `True`. Relate também o intervalo de confiança: se ele contém o zero, não afirme que a configuração é melhor.
- **Próximos passos:** trocar `DATASET`/dados pelo subconjunto do Quati (mesmo formato BEIR) e, para o português, usar o mE5 e o prompt traduzido.